# Week 10: Audio

Lists of numbers, but cooler.

In [ ]:
!wget -q https://github.com/PSAM-5005-2026S-A/5005-utils/raw/refs/heads/main/src/audio_utils.py

Let's import some libraries and helpers.

In [ ]:
import librosa
import matplotlib.pyplot as plt
import pandas as pd
import wave

from os import listdir
from random import randrange
from IPython.display import Audio

from audio_utils import list_to_wav
from audio_utils import fft, tone, multi_tone

## Digital Audio

Air pressure waves converted to electrical pulses, which are then sampled and turned into a sequence of numbers.

<img src="./imgs/audio-00.jpg" width="720" />

## Playing an audio file

is easy !

In [ ]:
display(Audio("./data/audio/two-bits.wav"))

### Look at the `data/` directory, and load and play some of the other files:

In [ ]:
# TODO: play other files
display(Audio("./data/audio/western.wav"))

## Audio file content

Instead of playing it we can get some information about our audio.

How long it is, how many channels it has...

In [ ]:
# open audio file for analysis and manipulation
wv = wave.open("./data/audio/two-bits.wav", mode="rb")

# print some info about the file
print(wv.getparams())

# Divide the number of frames (nframes) by framerate to get length in seconds

## Samples

In order to analyze and process our audio file we have to get to its samples: the numbers that represent the electric signals or pressure waves of the sound that was recorded.

<img src="./imgs/audio-01.jpg" width="720" />

We can sort of get to the samples using the `wave` object:

In [ ]:
print(wv.readframes(wv.getnframes()))

# but, this is writing out the file bytes as text... very cryptic and not very useful

## Librosa

This is a more robust library for working with audio and audio files.

We'll use it to load the audio portion of audio files into `Python` lists.

In [ ]:
# use library function to get decoded list of sample values
samples, sr = librosa.load("./data/audio/two-bits.wav", sr=None)

print(sr)

# print number of samples and first 10 samples
display(len(samples))
print(samples[:10])

# Can still play it using display(), but have to pass in the sample_rate
display(Audio(samples, rate=sr))

## "See" Audio

Once we have the samples we can plot it to see how it changes over time.

In [ ]:
plt.plot(samples, linestyle="-", marker="o", markersize=1)
# plt.xlim(5_500, 8_000)
plt.show()

# We'll be able to clearly see when it gets louder and softer (or, when notes are being played)

## Pure Tones

Let's take a little detour from audio files to just explore audio waves and their organization.

The simplest kind of sound is crated by a pure, oscillating, sine wave.

We can create a pure tone sound by calling the helper function `tone()` with a frequency value as the parameter.

In [ ]:
# Let's LOOK at some simple sounds

y, sr = tone(440, 2)

display(Audio(y, rate=sr))

plt.plot(y)
plt.show()

plt.plot(y)
plt.xlim([0, 1200])
plt.show()

# this wave goes up and down 440 times per second

## FFT

The Fourier Transform is kind of the opposite of the `tone()` function.

If we see the `tone()` function as something that turns a frequency value into a sound of that frequency, the `fft()` function can help us turn an arbitrary audio signal into a list of frequencies.


In [ ]:
# The FFT of our tone(440)

# this gets two lists, list of frequencies present in the signal (fp) and how powerful they are (fp)
fp, ff = fft(y, rate=sr)

plt.plot(ff, fp, linestyle="", marker="o", markersize=2)
plt.show()

# Zooming in
# plt.plot(ff, fp)
plt.plot(ff, fp, linestyle="", marker="o", markersize=2)
plt.xlim([350,500])
plt.show()

# we should see a "spike" at 440 Hz

### Why FFT ?

Well, it might not be super useful in these very simple situations, but audio is rarely just a single frequency.

The FFT operation can _decompose_ _*any*_ signal into a list of its constituent frequencies

In [ ]:
# Let's create a more complex sound made up of 4 separate frequencies
tones, sr = multi_tone([250, 440, 1400, 2500], length_seconds=2)

display(Audio(tones, rate=sr))

plt.plot(tones)
plt.show()

# Zooming in
plt.plot(tones)
plt.xlim([0, 500])
plt.show()

And if we use the `fft()` function, our list of power values should have peaks for each frequency of the individual sine waves that make up our sound.

In [ ]:
fp, ff = fft(tones)

plt.plot(ff, fp)
plt.show()

# Zooming in
plt.plot(ff, fp)
plt.xlim([100,3500])
plt.show()

## Back to Files

In [ ]:
samples, sr = librosa.load("./data/audio/two-bits.wav", sr=None)

display(Audio("./data/audio/two-bits.wav"))

plt.plot(samples)
plt.show()

# Zoom in
plt.plot(samples)
plt.xlim([18250, 20000])
plt.show()

### What About Frequencies ?

<img style="padding:10px; background:#fff;" src="https://upload.wikimedia.org/score/0/0/00h4jh1xffegoo127sdk0wur8p9w1mk/00h4jh1x.png" height="70px">

https://en.wikipedia.org/wiki/Shave_and_a_Haircut

In [ ]:
fp, ff = fft(samples)

plt.plot(ff, fp)
plt.show()

# Zooming in
plt.plot(ff, fp)
plt.xlim([100,1500])
plt.show()

## Look at Another File

Play the audio, and look at its wave and frequency plots.

This is the score for the `western.wav` file:

<div style="width: 800px; height: 150px; overflow: hidden;">
  <img src="https://musescore.com/static/musescore/scoredata/g/c6f7a21cfa356eeee2c8dd155feb97a53a867740/score_0.svg?no-cache=1715699146" style="width:100%; height:100%; object-fit: cover; object-position: 0% 11%; margin-left:-50px">
</div>

In [ ]:
# TODO: another file
#       wave
#       fft
osamp, sr = librosa.load("./data/audio/western.wav", sr=None)

## Manipulation

Just like we can directly manipulate pixel values to change the look of an image, we can also manipulate sample values in order to change the sound of an audio.

There are two basic aspects of an audio we can change this way: its amplitude (loudness), and its perceived speed (pitch).

If we increase sample values we make the sound louder.

### Note:

The notebook audio player tries to be smart and, when given raw samples, will scale our sound so it's never too loud or too soft. In order to hear the change in amplitude we first have to write out the audio to a file and then play the file.

We an write out the samples to a file with the `list_to_wav()` function. It takes a list of samples, their sample rate and the file name.

In [ ]:
display(Audio(samples, rate=sr))
samples[:10]

In [ ]:
# change sample amplitude

nsamples = []

for s in samples:
  nsamples.append(4 * s)

# display(Audio(nsamples, rate=sr))

list_to_wav(nsamples, sr, "loud.wav")
display(Audio("./loud.wav"))

In [ ]:
# TODO: change sample amplitude and make it softer

If we add (or remove) samples, we'll make the audio last longer (or shorter), but since the overall content of the audio is pretty much the same, the audio will seem slow (or fast).

Image removing every other letter from a sentence:

This sentence is now shorter $\rightarrow$ Ti snec i nw sotr

Or repeating every letter:

This sentence is now longer $\rightarrow$ Thhiiss sseenntteennccee iiss nnooww lloonnggeerr

Adding or removing a small number of samples is like adding or removing a small number of pixels to an image; we can increase and shrink the size of an image, and it will be distorted, but the overall content is still there. It just changes how we perceive it.

In [ ]:
# to remove every other sample from the list, we can iterate over samples
#   and append the ones from odd indexes

nsamples = []

for idx,s in enumerate(samples):
  if idx % 2 == 0:
    nsamples.append(s)

nsamples = samples[::2]

display(Audio(nsamples, rate=sr))

In [ ]:
# TODO: add repeated samples to make the audio longer
# TODO:   iterate over the samples and just append each twice

## Features

Sometimes we have to transform our data in order to get meaningful information from it.

For example, when analyzing audio, just sample values and $FFT$ plots are not enough to give us a concise description of our audio content.

We can be a bit more methodical about how we extract meaningful frequency and loudness values from our audio in a way that helps us organize and quickly compare different audio files.

### Shazam

This is what's behind the [Shazam algorithm](https://www.youtube.com/watch?v=b6xeOLjeKs0): a large database of songs that are described with only a few features each, so they can be organized, searched, queried and compared very very quickly.

Another [video](https://www.youtube.com/watch?v=kMNSAhsyiDg) reference.

## Librosa Features

Besides having functions to load audio files, the `librosa` library also has functions that help us analyze audio.

Let's see how to use these.

In [ ]:
# Let's open and play a song

y, sr = librosa.load("./data/audio/western.wav")
display(Audio(y, rate=sr))

In [ ]:
# TODO: plot waveform using plt.plot()
plt.plot(y)
plt.show()

### Root-Mean-Square Amplitude

This feature is a moving average of our audio signal. It basically reduces the number of samples we use to represent our audio.

We call it by passing a list of samples to `librosa.feature.rms()`.

Because these functions were written to work with multi-channel audio, they'll always return a list of lists. In our case, we just have a single channel, so our result will be at index $0$ of the returned list.

In [ ]:
len(y)

In [ ]:
# using [0] to get the first (and only) channel of the result
rms = librosa.feature.rms(y=y)[0]

print(len(rms))

# TODO: visualize rms signal
plt.plot(rms)
plt.plot([0, 598], [rms.mean(), rms.mean()])
plt.show()

### Frequency Features

We can also extract frequency information using `librosa`.

This is the numeric information that we looked at in the `FFT` plots.

Let's start by just visualizing the change in frequency over the duration of the audio.

In [ ]:
# Average frequency of our song

fcenter = librosa.feature.spectral_centroid(y=y)[0]

print(len(fcenter))

plt.plot(fcenter)
plt.ylim(0, 2000)
plt.show()

### Other Frequencies

Let's repeat the above plot, but now extract the lowest and highest frequencies for our signal.

We can use the librosa `spectral_rolloff()` function, and give it a `roll_percent` value of $0.01$ for extracting the lowest frequencies and $0.99$ for the highest frequencies.

In [ ]:
# Extract lowest and highest frequencies

sc = librosa.feature.spectral_centroid(y=y)[0]
fmin = librosa.feature.spectral_rolloff(y=y, roll_percent=0.01)[0]
fmax = librosa.feature.spectral_rolloff(y=y, roll_percent=0.99)[0]

plt.plot(sc)
plt.plot(fmin)
plt.plot(fmax)
plt.show()

# 🤔

This is a start.

We can average the `spectral_centroid` and the two `spectral_rolloff` frequencies to get $3$ values that represent our audio file. If these combinations of $3$ numbers are unique enough we have a way to create a _fingerprint_ for each audio file and use that to recognize song clips later.

We can extend this idea with Mel-Frequency Cepstral Coefficients (MFCC). MFCCs are a compact, numerical representation of an audio signal's full frequency characteristics. It divides an audio's frequency range into bands based on human hearing sensitivities and considers not just center, min and max values, but values for $10$ to $20$ different frequency sub-ranges.

The center frequency, min and max will still be there, but we'll also see information about other frequency bands ($2^{\text{nd}}$ highest frequency, $2^{\text{nd}}$ lowest, etc).

In [ ]:
mfccs = librosa.feature.mfcc(y=y, n_mfcc=10)

len(mfccs), len(mfccs[0])

for vs in mfccs:
  plt.plot(vs)

plt.show()

That's a lot of values.

Let's average each one to get a single value for each frequency band.

This way we can describe our song with just $10$ MFCCs ($10$ features).

In [ ]:
# Average the MFCCs, store in an array and create DataFrame

mfccs = librosa.feature.mfcc(y=y, n_mfcc=10)

mfcc_avgs = []

for vs in mfccs:
  mfcc_avg_val = vs.mean()
  mfcc_avgs.append(mfcc_avg_val)

mfccs_df = pd.DataFrame([mfcc_avgs], columns=[f"MFCC{i}" for i in range(len(mfccs))])
mfccs_df